# 장기 기억: Fact

`add_fact()`로 일반 사실과 유효기간을 갖는 사실을 저장하고, 저장된 내용을 그래프로 확인한다.
Fact는 주어·술어·목적어를 속성으로 갖는 독립된 노드이며, 신뢰도와 유효기간을 함께 저장할 수 있다.

실행 흐름은 클라이언트 연결, 일반 Fact 저장, 유효기간 Fact 저장, 시각화, 데이터 삭제, 연결 종료 순서이다.
[엔티티와 관계 실험](02_long_term_memory_entity.ipynb)의 실행 결과에 의존하지 않으며, 주어와 목적어에 대응하는 엔티티를 미리 생성할 필요가 없다.

## 실행 준비

로그 출력을 설정하고 공통 설정 모듈에서 `MemorySettings`를 구성한다.
클라이언트를 한 번 연결한 뒤 저장·시각화·삭제 단계에서 재사용한다. 각 노트북은 별도의 커널에서 독립적으로 실행할 수 있다.

In [ ]:
import logging

logging.getLogger("neo4j.notifications").setLevel(logging.ERROR)
logging.getLogger("transformers.configuration_utils").setLevel(logging.ERROR)

In [ ]:
from config.memory_client import load_memory_settings, print_memory_settings
from neo4j_agent_memory import MemorySettings

settings: MemorySettings = load_memory_settings()
print_memory_settings(settings)

In [ ]:
from neo4j_agent_memory import MemoryClient

memory_client = MemoryClient(settings)

try:
    await memory_client.connect()
except:
    await memory_client.close()
    raise

print("✓ Memory client opened")

## 일반 Fact 저장

`add_fact(subject, predicate, obj)`에 대상, 속성 또는 관계, 값을 전달하여 사실을 저장한다.
`confidence`는 입력 사실의 신뢰도를 나타낸다. 중복 제거가 활성화된 경우 기존 Fact가 재사용될 수 있다.

In [ ]:
facts = [
    # 사람 → 장소
    ("jeeklee", "lives_in", "성수", 0.9),
    ("jeeklee", "works_in", "성수", 0.8),

    # 사람 → 직무/조직
    ("jeeklee", "has_role", "Forward Deployed Engineer", 1.0),
    ("jeeklee", "works_at", "<회사명>", 1.0),

    # 사람 → 업무 맥락
    ("jeeklee", "supports_customer", "<고객사명>", 0.9),
    ("jeeklee", "specializes_in", "LLM 에이전트 도입", 0.7),

    # 사람 → 선호/습관 (대화에서 추론한 건 confidence 낮게)
    ("jeeklee", "prefers_language", "Korean", 0.9),
    ("jeeklee", "prefers", "간결한 코드 예시", 0.6),

    # 장소 → 장소 (계층)
    ("성수", "located_in", "성동구", 1.0),
    ("성동구", "located_in", "서울특별시", 1.0),
]

for subject, predicate, obj, confidence in facts:
    await memory_client.long_term.add_fact(
        subject=subject,
        predicate=predicate,
        obj=obj,
        confidence=confidence,
    )

## 유효기간을 포함한 Fact 저장

시간대 정보가 있는 `datetime`으로 `valid_from`과 `valid_until`을 지정한다. 종료 시점이 정해지지 않은 경우 `valid_until=None`을 사용한다.
아래 출력은 입력 목록에 지정한 유효기간을 표시한다. 중복 제거로 기존 Fact가 재사용되면 입력한 유효기간이 그대로 반영되지 않을 수 있으므로, 다음 시각화 단계에서 DB에 저장된 속성을 확인한다.

In [ ]:
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo

temporal_facts = [
    # 거주
    {
        "subject": "jeeklee",
        "predicate": "lives_in",
        "object": "망원",
        "valid_from": datetime(2021, 4, 1, tzinfo=ZoneInfo("Asia/Seoul")),
        "valid_until": datetime(2024, 2, 1, tzinfo=ZoneInfo("Asia/Seoul")),   # 이사 감
        "confidence": 0.9,
    },
    {
        "subject": "jeeklee",
        "predicate": "lives_in",
        "object": "성수",
        "valid_from": datetime(2024, 2, 1, tzinfo=ZoneInfo("Asia/Seoul")),
        "valid_until": None,
        "confidence": 0.9,
    },
    # 직무 변화
    {
        "subject": "jeeklee",
        "predicate": "has_role",
        "object": "Backend Engineer",
        "valid_from": datetime(2021, 3, 2, tzinfo=ZoneInfo("Asia/Seoul")),
        "valid_until": datetime(2025, 2, 1, tzinfo=ZoneInfo("Asia/Seoul")),
        "confidence": 1.0,
    },
    {
        "subject": "jeeklee",
        "predicate": "has_role",
        "object": "Forward Deployed Engineer",
        "valid_from": datetime(2025, 3, 4, tzinfo=ZoneInfo("Asia/Seoul")),
        "valid_until": None,
        "confidence": 1.0,
    },
    # 도구 / 학습
    {
        "subject": "jeeklee",
        "predicate": "uses",
        "object": "long-term memory client",
        "valid_from": datetime(2026, 11, 9, tzinfo=ZoneInfo("Asia/Seoul")),
        "valid_until": None,
        "confidence": 0.7,
    },
    {
        "subject": "jeeklee",
        "predicate": "studies",
        "object": "지식 그래프 설계",
        "valid_from": datetime(2026, 9, 1, tzinfo=ZoneInfo("Asia/Seoul")),
        "valid_until": None,
        "confidence": 0.6,
    },
]

for fact in temporal_facts:
    await memory_client.long_term.add_fact(
        subject=fact["subject"],
        predicate=fact["predicate"],
        obj=fact["object"],
        valid_from=fact["valid_from"],
        valid_until=fact["valid_until"],
        confidence=fact["confidence"],
    )
    validity = "VALID" \
        if fact["valid_until"] is None \
        else f"NOT_VALID ({fact['valid_from'].strftime('%Y-%m-%d')} ~ {fact['valid_until'].strftime('%Y-%m-%d')})"
    print(f"✓ {fact['subject']} -[{fact['predicate']}]-> {fact['object']}")
    print(f"  {validity}")

## 생성한 Fact 시각화

일반 사실 목록 `facts`와 유효기간을 포함한 `temporal_facts`를 기준으로 DB의 Fact를 조회하고, 공유 모듈 `graph_visualizer`로 시각화한다.
각 Fact는 하나의 독립된 노드로 표시되며, 캡션에는 주어·술어·목적어의 실제 값이 나타난다. 노드 속성에서 신뢰도와 저장된 유효기간을 확인한다.
`add_fact()`는 엔티티 간 관계를 생성하지 않으므로 Fact 사이의 연결선은 표시되지 않는다.

조회 기준은 입력한 세 필드의 일치이며, 같은 값을 가진 기존 Fact도 포함된다. `DISTINCT`는 같은 노드가 중복 조회되는 것을 방지한다.
중복 제거로 기존 Fact가 재사용된 경우 저장된 유효기간은 입력값과 다를 수 있다.
조회 결과 `stored_facts`의 ID는 시각화 대상과 다음 삭제 셀의 대상을 지정하는 데 함께 사용한다.


In [ ]:
from importlib import reload
from typing import LiteralString

import graph_visualizer

# 실행 중인 커널에 공유 모듈의 최신 캡션 설정을 반영한다.
reload(graph_visualizer)

fact_patterns = [
    {"subject": subject, "predicate": predicate, "object": obj}
    for subject, predicate, obj, _ in facts
] + [
    {key: fact[key] for key in ("subject", "predicate", "object")}
    for fact in temporal_facts
]

fact_query: LiteralString = """
UNWIND $facts AS item
MATCH (f:Fact {
    subject: item.subject,
    predicate: item.predicate,
    object: item.object
})
RETURN DISTINCT f.id AS id,
       f.subject AS subject, f.predicate AS predicate, f.object AS object,
       f.confidence AS confidence,
       f.valid_from AS valid_from, f.valid_until AS valid_until
ORDER BY subject, predicate, valid_from, object, id
"""

fact_graph_query: LiteralString = """
MATCH (f:Fact)
WHERE f.id IN $fact_ids
RETURN f
"""

async with memory_client.graph.session() as graph_session:
    result = await graph_session.run(fact_query, {"facts": fact_patterns})
    stored_facts = await result.data()

    print(f"✓ Stored facts: {len(stored_facts)}")
    await graph_visualizer.visualize(
        graph_session,
        fact_graph_query,
        {"fact_ids": [fact["id"] for fact in stored_facts]},
    )


## 실험 데이터 삭제

앞선 시각화 셀의 `stored_facts`에 포함된 ID의 Fact를 삭제하고 삭제 건수를 출력한다.
시각화 대상에는 입력한 주어·술어·목적어와 값이 같은 기존 Fact도 포함되므로 조회 결과를 확인한 뒤 실행한다.
데이터를 보존하는 경우 이 셀을 생략하고 클라이언트 종료 셀만 실행한다.

In [ ]:
from typing import LiteralString

cleanup_query: LiteralString = """
MATCH (f:Fact)
WHERE f.id IN $fact_ids
DETACH DELETE f
RETURN count(*) AS deleted_facts
"""

async with memory_client.graph.session() as graph_session:
    result = await graph_session.run(
        cleanup_query,
        {"fact_ids": [fact["id"] for fact in stored_facts]},
    )
    deleted = await result.single(strict=True)
    await result.consume()

print(f"✓ Deleted facts: {deleted['deleted_facts']}")

## 메모리 클라이언트 종료

`close()`로 메모리 클라이언트의 연결과 백그라운드 작업을 정리한다.
연결 종료는 DB에 저장된 데이터를 삭제하지 않으므로, 데이터 삭제 여부와 관계없이 실험을 마칠 때 실행한다.
종료 후 실험을 재개하려면 클라이언트 생성 및 연결 셀부터 다시 실행한다.

In [ ]:
await memory_client.close()
print("✓ Memory client closed")